# 05 — Build Gold

## Objective

Build the analytical Gold layer from cleaned and validated Silver tables.

## This notebook performs

- Reading of Silver Delta tables.
- Creation of analytical dimensions:
  - `dim_region`
  - `dim_date`
  - `dim_employee`
  - `dim_client`
- Creation of analytical fact tables:
  - `fact_workforce_monthly`
  - `fact_goals_monthly`
- Generation of model-owned surrogate keys.
- Preservation of source identifiers from Silver for traceability.
- Calculation of executive workforce KPIs.

## Notes

The Gold layer is designed for reporting and dashboard consumption.

Silver keeps clean source identifiers such as `employee_id`, `client_id` and `region_id`.

Gold creates model-owned keys such as `employee_key`, `client_key`, `region_key` and `date_key`, which are used to connect dimensions and fact tables.

In [0]:
# Import Spark functions used to build dimensions, fact tables and surrogate keys.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Use the project schema where Silver tables were created.

spark.sql("USE SCHEMA workbridge")


DataFrame[]

In [0]:
# Read Silver tables created and validated in previous notebooks.

silver_employees_df = spark.table("silver_employees")
silver_clients_df = spark.table("silver_clients")
silver_assignments_df = spark.table("silver_assignments")
silver_hours_df = spark.table("silver_hours")
silver_costs_df = spark.table("silver_costs")
silver_goals_df = spark.table("silver_goals")

# Create dim_region

## Objective

Create the region dimension used by employees, clients and fact tables.

## Source

Regions are collected from the normalized `region_id` values available in Silver tables.

## Output

Gold table: `dim_region`

## Columns

- `region_key`: model-owned surrogate key.
- `region_id`: normalized business region code from Silver.
- `region_name`: descriptive region name.

## Notes

`region_id` is the normalized region code used in Silver, such as `LATAM`, `NA`, `EMEA` and `APAC`.

`region_key` is generated in Gold and will be used as a foreign key in dimensions and fact tables.

In [0]:
# Get valid region codes from employees.
employee_regions_df = (
    silver_employees_df
    .select("region_id")
    .where(F.col("region_id").isNotNull())
)

# Get valid region codes from clients.
client_regions_df = (
    silver_clients_df
    .select("region_id")
    .where(F.col("region_id").isNotNull())
)

# Get valid region codes from costs.
cost_regions_df = (
    silver_costs_df
    .select("region_id")
    .where(F.col("region_id").isNotNull())
)

# Get valid region codes from goals.
goal_regions_df = (
    silver_goals_df
    .select("region_id")
    .where(F.col("region_id").isNotNull())
)

# Combine all region codes and keep only unique values.
all_regions_df = (
    employee_regions_df
    .unionByName(client_regions_df)
    .unionByName(cost_regions_df)
    .unionByName(goal_regions_df)
    .distinct()
)

# Add a readable region name for each normalized region code.
region_name_df = (
    all_regions_df
    .withColumn(
        "region_name",
        F.when(F.col("region_id") == "LATAM", "Latin America")
         .when(F.col("region_id") == "NA", "North America")
         .when(F.col("region_id") == "EMEA", "Europe, Middle East and Africa")
         .when(F.col("region_id") == "APAC", "Asia Pacific")
         .otherwise("Unknown")
    )
)

# Define a stable ordering to generate region surrogate keys.
region_window = Window.orderBy("region_id")

# Create the final region dimension with a model-owned surrogate key.
dim_region_df = (
    region_name_df
    .withColumn("region_key", F.row_number().over(region_window))
    .select(
        "region_key",
        "region_id",
        "region_name"
    )
)

# Preview the Gold region dimension.
display(dim_region_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


region_key,region_id,region_name
1,APAC,Asia Pacific
2,EMEA,"Europe, Middle East and Africa"
3,LATAM,Latin America
4,NA,North America


In [0]:
# Check the result of dim_region creation.
# This quick validation checks row count and uniqueness of the region key and region code.

dim_region_count = dim_region_df.count()

print(f"Gold dim_region rows: {dim_region_count}")

# Check that region_key is unique.
region_key_duplicates = (
    dim_region_df
    .groupBy("region_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicated region_key values: {region_key_duplicates}")

# Check that region_id is unique.
region_id_duplicates = (
    dim_region_df
    .groupBy("region_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicated region_id values: {region_id_duplicates}")

# Check that region_name is not null.
region_name_null_count = (
    dim_region_df
    .filter(F.col("region_name").isNull())
    .count()
)

print(f"Rows with null region_name: {region_name_null_count}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Gold dim_region rows: 4
Duplicated region_key values: 0
Duplicated region_id values: 0
Rows with null region_name: 0


# Create dim_date

## Objective

Create the date dimension used by Gold fact tables.

## Source

Periods are collected from Silver tables that contain monthly data:

- `silver_hours`
- `silver_costs`
- `silver_goals`

## Output

Gold table: `dim_date`

## Columns

- `date_key`: model-owned date key in `YYYYMM` format.
- `period`: normalized monthly period in `YYYY-MM` format.
- `year`: year number.
- `quarter`: calendar quarter.
- `month`: month number.
- `month_name`: month name.

## Notes

The source data is monthly, so this dimension is built at monthly granularity.

In [0]:
# Get monthly periods from hours.
hours_periods_df = (
    silver_hours_df
    .select("period")
    .where(F.col("period").isNotNull())
)

# Get monthly periods from costs.
costs_periods_df = (
    silver_costs_df
    .select("period")
    .where(F.col("period").isNotNull())
)

# Get monthly periods from goals.
goals_periods_df = (
    silver_goals_df
    .select("period")
    .where(F.col("period").isNotNull())
)

# Combine all periods and keep only unique values.
all_periods_df = (
    hours_periods_df
    .unionByName(costs_periods_df)
    .unionByName(goals_periods_df)
    .distinct()
)

# Convert the period string YYYY-MM into a real date using the first day of the month.
date_base_df = (
    all_periods_df
    .withColumn("period_start_date", F.to_date(F.concat(F.col("period"), F.lit("-01"))))
)

# Create date attributes for reporting.
dim_date_df = (
    date_base_df
    .withColumn("date_key", F.date_format(F.col("period_start_date"), "yyyyMM").cast("int"))
    .withColumn("year", F.year(F.col("period_start_date")))
    .withColumn("month", F.month(F.col("period_start_date")))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter(F.col("period_start_date"))))
    .withColumn("month_name", F.date_format(F.col("period_start_date"), "MMMM"))
    .select(
        "date_key",
        "period",
        "year",
        "quarter",
        "month",
        "month_name"
    )
    .orderBy("date_key")
)

# Preview the Gold date dimension.
display(dim_date_df)

date_key,period,year,quarter,month,month_name
202601,2026-01,2026,Q1,1,January
202602,2026-02,2026,Q1,2,February
202603,2026-03,2026,Q1,3,March
202604,2026-04,2026,Q2,4,April
202605,2026-05,2026,Q2,5,May
202606,2026-06,2026,Q2,6,June
202607,2026-07,2026,Q3,7,July
202608,2026-08,2026,Q3,8,August
202609,2026-09,2026,Q3,9,September
202610,2026-10,2026,Q4,10,October


In [0]:
# Check the result of dim_date creation.
# This quick validation checks row count and uniqueness of the date key and period.

dim_date_count = dim_date_df.count()

print(f"Gold dim_date rows: {dim_date_count}")

# Check that date_key is unique.
date_key_duplicates = (
    dim_date_df
    .groupBy("date_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicated date_key values: {date_key_duplicates}")

# Check that period is unique.
period_duplicates = (
    dim_date_df
    .groupBy("period")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicated period values: {period_duplicates}")

# Check that date attributes are not null.
date_attributes_null_count = (
    dim_date_df
    .filter(
        F.col("date_key").isNull()
        | F.col("period").isNull()
        | F.col("year").isNull()
        | F.col("quarter").isNull()
        | F.col("month").isNull()
        | F.col("month_name").isNull()
    )
    .count()
)

print(f"Rows with null date attributes: {date_attributes_null_count}")

Gold dim_date rows: 12
Duplicated date_key values: 0
Duplicated period values: 0
Rows with null date attributes: 0


# Create dim_employee

## Objective

Create the employee dimension used by Gold fact tables.

## Source

Source table: `silver_employees`

## Output

Gold table: `dim_employee`

## Columns

- `employee_key`: model-owned numeric surrogate key.
- `employee_id`: source employee identifier from Silver.
- `employee_name`
- `employee_email`
- `email_is_valid`
- `gender`
- `country`
- `region_key`
- `department`
- `role`
- `seniority`
- `hire_date`
- `termination_date`
- `employment_status`
- `valid_from`
- `valid_to`
- `is_current`

## Identity and SCD readiness

`employee_id` is preserved as the original source identifier for traceability.

`employee_key` is generated in Gold as a model-owned numeric surrogate key and will be used as a foreign key in fact tables.

This dimension includes SCD Type 2 columns: `valid_from`, `valid_to` and `is_current`.

Since this is an initial Gold load and the current dataset contains a single cleaned employee snapshot, the notebook does not perform historical versioning yet. All employee records are loaded as current records.

In a future incremental load, SCD Type 2 logic would compare incoming employee records against existing current records and create a new version when tracked attributes such as department, role, seniority, region or employment status change.

In [0]:
# Join employees with dim_region to replace region_id with region_key.
employee_region_df = (
    silver_employees_df
    .join(
        dim_region_df.select("region_key", "region_id"),
        on="region_id",
        how="left"
    )
)

# Define a stable order before generating numeric surrogate keys.
employee_window = Window.orderBy(
    F.col("employee_id"),
    F.lower(F.trim(F.col("employee_name")))
)

# Create the final employee dimension.
dim_employee_df = (
    employee_region_df
    
    # Generate a model-owned numeric surrogate key.
    .withColumn("employee_key", F.row_number().over(employee_window))
    
    # Prepare SCD Type 2 columns.
    # This is an initial load, so every employee is treated as current.
    .withColumn("valid_from", F.col("hire_date"))
    .withColumn("valid_to", F.lit(None).cast("date"))
    .withColumn("is_current", F.lit(True))
    
    # Select the final Gold dimension columns.
    .select(
        "employee_key",
        "employee_id",
        "employee_name",
        "employee_email",
        "email_is_valid",
        "gender",
        "country",
        "region_key",
        "department",
        "role",
        "seniority",
        "hire_date",
        "termination_date",
        "employment_status",
        "valid_from",
        "valid_to",
        "is_current"
    )
)

# Preview the Gold employee dimension.
display(dim_employee_df.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


employee_key,employee_id,employee_name,employee_email,email_is_valid,gender,country,region_key,department,role,seniority,hire_date,termination_date,employment_status,valid_from,valid_to,is_current
1,E0001,Daniel Taylor,daniel.taylor.e0001@workbridge.com,true,Female,Australia,1,Engineering,HR Specialist,Semi Senior,2024-03-25,null,active,2024-03-25,null,true
2,E0002,Daniel Rodriguez,daniel.rodriguez.e0002@workbridge.com,true,Not specified,India,1,Customer Success,Data Analyst,Senior,2025-02-27,null,active,2025-02-27,null,true
3,E0003,Sofia Rodriguez,sofia.rodriguez.e0003@workbridge.com,true,Not specified,Spain,2,Data,HR Specialist,Senior,2024-08-24,null,active,2024-08-24,null,true
4,E0004,Daniel Garcia,daniel.garcia.e0004@workbridge.com,true,Not specified,United States,4,Data,Data Analyst,Senior,2025-02-04,null,active,2025-02-04,null,true
5,E0005,Olivia Martinez,olivia.martinez.e0005@workbridge.com,true,Not specified,Germany,2,HR,Data Engineer,Semi Senior,2023-04-29,null,active,2023-04-29,null,true
6,E0006,John Gonzalez,john.gonzalez.e0006@workbridge.com,true,Not specified,Japan,1,Operations,Project Manager,Senior,2024-06-26,null,active,2024-06-26,null,true
7,E0007,Ava Garcia,ava.garcia.e0007@workbridge.com,true,Not specified,Singapore,1,Engineering,Data Analyst,Junior,2023-09-05,null,active,2023-09-05,null,true
8,E0008,Olivia Ruiz,olivia.ruiz.e0008@workbridge.com,true,Female,Brazil,3,Engineering,Business Analyst,Semi Senior,2023-05-19,null,active,2023-05-19,null,true
9,E0009,Mia Johnson,mia.johnson.e0009@workbridge.com,true,Not specified,Japan,1,Finance,Project Manager,Lead,2025-08-26,null,active,2025-08-26,null,true
10,E0010,Mia Davis,mia.davis.e0010@workbridge.com,true,Female,Argentina,3,Engineering,Project Manager,Semi Senior,2023-05-05,null,active,2023-05-05,null,true


In [0]:
# Count records in Silver and Gold to confirm no employees were lost.
silver_employees_count = silver_employees_df.count()
dim_employee_count = dim_employee_df.count()

print(f"Silver employees rows: {silver_employees_count}")
print(f"Gold dim_employee rows: {dim_employee_count}")

# Check employees without region_key.
# This is expected for employees whose region was kept as warning-level in Silver.
employees_without_region_key_count = (
    dim_employee_df
    .filter(F.col("region_key").isNull())
    .count()
)

print(f"Employees without region_key: {employees_without_region_key_count}")

# Check that employee_key is unique.
employee_key_duplicates = (
    dim_employee_df
    .groupBy("employee_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicated employee_key values: {employee_key_duplicates}")

# Check that employee_id is unique in the current initial-load dimension.
employee_id_duplicates = (
    dim_employee_df
    .groupBy("employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicated employee_id values: {employee_id_duplicates}")

# Check that employee_key is not null.
employee_key_null_count = (
    dim_employee_df
    .filter(F.col("employee_key").isNull())
    .count()
)

print(f"Rows with null employee_key: {employee_key_null_count}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Silver employees rows: 493
Gold dim_employee rows: 493
Employees without region_key: 44
Duplicated employee_key values: 0
Duplicated employee_id values: 0
Rows with null employee_key: 0


# Create dim_client

## Objective

Create the client dimension used by Gold fact tables.

## Source

Source table: `silver_clients`

## Output

Gold table: `dim_client`

## Columns

- `client_key`: model-owned numeric surrogate key.
- `client_id`: source client identifier from Silver.
- `client_name`
- `standardized_client_name`
- `industry`
- `region_key`
- `account_manager`
- `account_manager_missing_flag`
- `contract_type`
- `potential_duplicate_flag`

## Notes

`client_id` is preserved as the original source identifier for traceability.

`client_key` is generated in Gold as a model-owned numeric surrogate key and will be used as a foreign key in fact tables.

`region_key` links each client to `dim_region`.

Potential duplicate clients are preserved because they were treated as warning-level issues in Silver, not critical rejections.

In [0]:
# Join clients with dim_region to replace region_id with region_key.
client_region_df = (
    silver_clients_df
    .join(
        dim_region_df.select("region_key", "region_id"),
        on="region_id",
        how="left"
    )
)

# Define a stable order before generating numeric surrogate keys.
client_window = Window.orderBy(
    F.col("client_id"),
    F.lower(F.trim(F.col("client_name")))
)

# Create the final client dimension.
dim_client_df = (
    client_region_df
    
    # Generate a model-owned numeric surrogate key.
    .withColumn("client_key", F.row_number().over(client_window))
    
    # Select the final Gold dimension columns.
    .select(
        "client_key",
        "client_id",
        "client_name",
        "standardized_client_name",
        "industry",
        "region_key",
        "account_manager",
        "account_manager_missing_flag",
        "contract_type",
        "potential_duplicate_flag"
    )
)

# Preview the Gold client dimension.
display(dim_client_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


client_key,client_id,client_name,standardized_client_name,industry,region_key,account_manager,account_manager_missing_flag,contract_type,potential_duplicate_flag
1,C001,Alpha Bank,alphabank,Banking,3,Lucas Lopez,false,Managed Service,true
2,C002,Nova Retail,novaretail,Retail,3,Sofia Martinez,false,Managed Service,true
3,C003,MediCore Health,medicorehealth,Banking,4,Michael Thomas,false,Managed Service,false
4,C004,TechNova Systems,technovasystems,Telecommunications,1,John Perez,false,Fixed Price,true
5,C006,IronWorks Manufacturing,ironworksmanufacturing,Telecommunications,2,Mia Sanchez,false,Fixed Price,false
6,C007,GreenGrid Energy,greengridenergy,Manufacturing,2,null,true,Managed Service,false
7,C009,Andes Finance,andesfinance,Healthcare,2,Mateo Martinez,false,Time and Materials,false
8,C010,BrightMart,brightmart,Technology,4,null,true,Fixed Price,false
9,C011,CloudPath Technologies,cloudpathtechnologies,Banking,2,Valentina Rodriguez,false,Fixed Price,false
10,C012,PrimeCare Services,primecareservices,Manufacturing,4,James Lopez,false,Fixed Price,false


In [0]:
# Count records in Silver and Gold to confirm no clients were lost.
silver_clients_count = silver_clients_df.count()
dim_client_count = dim_client_df.count()

print(f"Silver clients rows: {silver_clients_count}")
print(f"Gold dim_client rows: {dim_client_count}")

# Check clients without region_key.
# This should be 0 because clients with invalid region were rejected in Silver.
clients_without_region_key_count = (
    dim_client_df
    .filter(F.col("region_key").isNull())
    .count()
)

print(f"Clients without region_key: {clients_without_region_key_count}")

# Check that client_key is unique.
client_key_duplicates = (
    dim_client_df
    .groupBy("client_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicated client_key values: {client_key_duplicates}")

# Check that client_id is unique in the Gold dimension.
client_id_duplicates = (
    dim_client_df
    .groupBy("client_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicated client_id values: {client_id_duplicates}")

# Check that client_key is not null.
client_key_null_count = (
    dim_client_df
    .filter(F.col("client_key").isNull())
    .count()
)

print(f"Rows with null client_key: {client_key_null_count}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Silver clients rows: 13
Gold dim_client rows: 13
Clients without region_key: 0
Duplicated client_key values: 0
Duplicated client_id values: 0
Rows with null client_key: 0


# Create fact_workforce_monthly

## Objective

Create the main workforce fact table for monthly executive reporting.

## Sources

- `silver_hours`
- `silver_costs`
- `dim_date`
- `dim_employee`
- `dim_client`
- `dim_region`

## Output

Gold table: `fact_workforce_monthly`

## Grain

One row per:

- monthly period
- employee
- client
- project

This means each row represents the monthly workforce activity of one employee for one client and one project.

## Keys

- `workforce_fact_key`: model-owned numeric surrogate key for each fact record.
- `date_key`: links the record to `dim_date`.
- `employee_key`: links the record to `dim_employee`.
- `client_key`: links the record to `dim_client`.
- `region_key`: links the record to `dim_region`.

## Source identifiers preserved for traceability

- `hours_record_id`
- `period`
- `employee_id`
- `client_id`
- `project_id`

These identifiers come from Silver and are kept to trace Gold records back to their cleaned source data.

## Metrics

- `available_hours`: total hours the employee was available to work during the period.
- `worked_hours`: total hours actually worked by the employee during the period.
- `billable_hours`: hours that can be billed to the client.
- `non_billable_hours`: worked hours that cannot be billed to the client.
- `overtime_hours`: hours worked beyond the expected available hours.
- `salary_cost`: salary cost associated with the employee for the period.
- `benefits_cost`: additional employment-related cost for the period.
- `total_cost`: total employee cost for the period, calculated as salary cost plus benefits cost.
- `utilization_rate`: percentage of available hours that were actually worked.
- `billable_rate`: percentage of worked hours that were billable.

## KPI formulas

- `utilization_rate = worked_hours / available_hours`
- `billable_rate = billable_hours / worked_hours`

## Notes

`region_key` is based on the client region, because executive workforce reporting is usually analyzed by client or commercial region.

Costs are joined at employee-month level using `period + employee_id`, while hours are recorded at `period + employee + client + project` level. This means employee monthly costs may repeat across multiple client/project hour records if an employee worked for more than one client or project in the same month.

`utilization_rate` can be greater than 1 when worked hours exceed available hours due to overtime.

For this portfolio version, this approach is accepted to enable combined workforce and cost reporting. A future improvement could allocate monthly employee costs proportionally based on worked hours.

In [0]:
# Start from monthly hour records because hours define the main fact grain.
workforce_base_df = silver_hours_df.alias("h")

# Add date_key from dim_date using the monthly period.
workforce_with_date_df = (
    workforce_base_df
    .join(
        dim_date_df.select("date_key", "period").alias("d"),
        on="period",
        how="left"
    )
)

# Add employee_key from dim_employee using the source employee_id.
workforce_with_employee_df = (
    workforce_with_date_df
    .join(
        dim_employee_df.select("employee_key", "employee_id").alias("e"),
        on="employee_id",
        how="left"
    )
)

# Add client_key and client region_key from dim_client using the source client_id.
workforce_with_client_df = (
    workforce_with_employee_df
    .join(
        dim_client_df.select(
            "client_key",
            "client_id",
            F.col("region_key").alias("client_region_key")
        ).alias("c"),
        on="client_id",
        how="left"
    )
)

# Add monthly employee costs using period + employee_id.
# Costs are at employee-month grain.
# Hours are at period + employee + client + project grain.
workforce_with_costs_df = (
    workforce_with_client_df
    .join(
        silver_costs_df
        .select(
            "period",
            "employee_id",
            "salary_cost",
            "benefits_cost",
            "total_cost"
        )
        .alias("co"),
        on=["period", "employee_id"],
        how="left"
    )
)

# Define a stable order before generating the numeric surrogate key for the fact.
workforce_fact_window = Window.orderBy("hours_record_id")

# Create the final workforce fact table.
fact_workforce_monthly_df = (
    workforce_with_costs_df
    
    # Generate a model-owned numeric surrogate key for each fact record.
    .withColumn("workforce_fact_key", F.row_number().over(workforce_fact_window))
    
    # Calculate utilization rate.
    .withColumn(
        "utilization_rate",
        F.when(F.col("available_hours") > 0, F.col("worked_hours") / F.col("available_hours"))
         .otherwise(None)
    )
    
    # Calculate billable rate.
    .withColumn(
        "billable_rate",
        F.when(F.col("worked_hours") > 0, F.col("billable_hours") / F.col("worked_hours"))
         .otherwise(None)
    )
    
    # Select the final Gold fact columns.
    .select(
        "workforce_fact_key",
        "hours_record_id",
        "date_key",
        "employee_key",
        "client_key",
        F.col("client_region_key").alias("region_key"),
        "period",
        "employee_id",
        "client_id",
        "project_id",
        "available_hours",
        "worked_hours",
        "billable_hours",
        "non_billable_hours",
        "overtime_hours",
        "salary_cost",
        "benefits_cost",
        "total_cost",
        "utilization_rate",
        "billable_rate",
        "source_file",
        "ingestion_timestamp",
        "bronze_load_id"
    )
)

# Preview the main Gold fact table.
display(fact_workforce_monthly_df.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


workforce_fact_key,hours_record_id,date_key,employee_key,client_key,region_key,period,employee_id,client_id,project_id,available_hours,worked_hours,billable_hours,non_billable_hours,overtime_hours,salary_cost,benefits_cost,total_cost,utilization_rate,billable_rate,source_file,ingestion_timestamp,bronze_load_id
1,2026-01|E0001|C007|P016,202601,1,6,2,2026-01,E0001,C007,P016,160.0,103.0,95.0,8.0,0.0,2705.69,333.17,3038.86,0.64375,0.9223300970873787,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2,2026-01|E0002|C004|P023,202601,2,4,1,2026-01,E0002,C004,P023,160.0,146.0,128.0,18.0,0.0,5095.99,1025.58,6121.57,0.9125,0.8767123287671232,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
3,2026-01|E0003|C012|P035,202601,3,10,4,2026-01,E0003,C012,P035,160.0,168.0,111.0,57.0,8.0,5714.16,912.83,6626.99,1.05,0.6607142857142857,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
4,2026-01|E0006|C001|P007,202601,6,1,3,2026-01,E0006,C001,P007,160.0,160.0,111.0,49.0,0.0,4217.11,836.34,5053.45,1.0,0.69375,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
5,2026-01|E0007|C010|P018,202601,7,8,4,2026-01,E0007,C010,P018,160.0,144.0,123.0,21.0,0.0,2132.64,359.75,2492.39,0.9,0.8541666666666666,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
6,2026-01|E0009|C004|P034,202601,9,4,1,2026-01,E0009,C004,P034,160.0,156.0,129.0,27.0,0.0,7781.96,1319.32,9101.28,0.975,0.8269230769230769,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
7,2026-01|E0010|C011|P001,202601,10,9,2,2026-01,E0010,C011,P001,160.0,116.0,79.0,37.0,0.0,4051.69,778.24,4829.93,0.725,0.6810344827586207,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
8,2026-01|E0011|C002|P029,202601,11,2,3,2026-01,E0011,C002,P029,160.0,186.0,171.0,15.0,26.0,1830.76,221.38,2052.14,1.1625,0.9193548387096774,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
9,2026-01|E0012|C004|P003,202601,12,4,1,2026-01,E0012,C004,P003,160.0,148.0,106.0,42.0,0.0,8278.41,1655.85,9934.26,0.925,0.7162162162162162,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
10,2026-01|E0014|C011|P005,202601,14,9,2,2026-01,E0014,C011,P005,160.0,180.0,150.0,30.0,20.0,2069.74,270.02,2339.76,1.125,0.8333333333333334,hours.csv,2026-05-09T12:44:18.627Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


In [0]:
# Count records in Silver hours and Gold fact_workforce_monthly.
silver_hours_count = silver_hours_df.count()
fact_workforce_count = fact_workforce_monthly_df.count()

print(f"Silver hours rows: {silver_hours_count}")
print(f"Gold fact_workforce_monthly rows: {fact_workforce_count}")

# Check that workforce_fact_key is unique.
workforce_fact_key_duplicates = (
    fact_workforce_monthly_df
    .groupBy("workforce_fact_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicated workforce_fact_key values: {workforce_fact_key_duplicates}")

# Check that workforce_fact_key is not null.
workforce_fact_key_null_count = (
    fact_workforce_monthly_df
    .filter(F.col("workforce_fact_key").isNull())
    .count()
)

print(f"Rows with null workforce_fact_key: {workforce_fact_key_null_count}")

# Check missing dimensional keys.
missing_date_key_count = fact_workforce_monthly_df.filter(F.col("date_key").isNull()).count()
missing_employee_key_count = fact_workforce_monthly_df.filter(F.col("employee_key").isNull()).count()
missing_client_key_count = fact_workforce_monthly_df.filter(F.col("client_key").isNull()).count()
missing_region_key_count = fact_workforce_monthly_df.filter(F.col("region_key").isNull()).count()

print(f"Rows without date_key: {missing_date_key_count}")
print(f"Rows without employee_key: {missing_employee_key_count}")
print(f"Rows without client_key: {missing_client_key_count}")
print(f"Rows without region_key: {missing_region_key_count}")

# Check that utilization_rate is within a reasonable range.
# Values above 1 are possible when overtime hours exist.
invalid_utilization_rate_count = (
    fact_workforce_monthly_df
    .filter((F.col("utilization_rate") < 0) | (F.col("utilization_rate") > 2.0))
    .count()
)

invalid_billable_rate_count = (
    fact_workforce_monthly_df
    .filter((F.col("billable_rate") < 0) | (F.col("billable_rate") > 1))
    .count()
)

print(f"Rows with invalid utilization_rate: {invalid_utilization_rate_count}")
print(f"Rows with invalid billable_rate: {invalid_billable_rate_count}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Silver hours rows: 3292
Gold fact_workforce_monthly rows: 3292
Duplicated workforce_fact_key values: 0
Rows with null workforce_fact_key: 0
Rows without date_key: 0
Rows without employee_key: 0
Rows without client_key: 0
Rows without region_key: 0
Rows with invalid utilization_rate: 0
Rows with invalid billable_rate: 0


# Create fact_goals_monthly

## Objective

Create the monthly goals fact table used to compare workforce performance against business targets.

## Source

Source table: `silver_goals`

## Output

Gold table: `fact_goals_monthly`

## Grain

One row per:

- monthly period
- client
- region

This means each row represents the monthly target values defined for one client in one region.

## Keys

- `goals_fact_key`: model-owned numeric surrogate key for each fact record.
- `date_key`: links the record to `dim_date`.
- `client_key`: links the record to `dim_client`.
- `region_key`: links the record to `dim_region`.

## Source identifiers preserved for traceability

- `goal_record_id`
- `period`
- `client_id`
- `region_id`

These identifiers come from Silver and are kept to trace Gold records back to their cleaned source data.

## Metrics

- `target_turnover_rate`: expected or acceptable employee turnover rate for the period.
- `target_utilization_rate`: expected utilization rate for the client and region.
- `target_cost`: expected workforce cost for the client and region during the period.
- `target_billable_hours`: expected number of billable hours for the client and region during the period.

## Notes

This fact table is kept separate from `fact_workforce_monthly` because goals have a different grain.

`fact_workforce_monthly` is at period + employee + client + project level, while `fact_goals_monthly` is at period + client + region level.

Keeping goals separate avoids duplicating target values across multiple employee or project records.

In [0]:
# Start from Silver goals because it defines the grain of the goals fact.
goals_base_df = silver_goals_df.alias("g")

# Add date_key from dim_date using the monthly period.
goals_with_date_df = (
    goals_base_df
    .join(
        dim_date_df.select("date_key", "period").alias("d"),
        on="period",
        how="left"
    )
)

# Add client_key from dim_client using the source client_id.
goals_with_client_df = (
    goals_with_date_df
    .join(
        dim_client_df.select("client_key", "client_id").alias("c"),
        on="client_id",
        how="left"
    )
)

# Add region_key from dim_region using the normalized region_id.
goals_with_region_df = (
    goals_with_client_df
    .join(
        dim_region_df.select("region_key", "region_id").alias("r"),
        on="region_id",
        how="left"
    )
)

# Define a stable order before generating the numeric surrogate key for the fact.
goals_fact_window = Window.orderBy("goal_record_id")

# Create the final goals fact table.
fact_goals_monthly_df = (
    goals_with_region_df
    
    # Generate a model-owned numeric surrogate key for each goals fact record.
    .withColumn("goals_fact_key", F.row_number().over(goals_fact_window))
    
    # Select the final Gold fact columns.
    .select(
        "goals_fact_key",
        "goal_record_id",
        "date_key",
        "client_key",
        "region_key",
        "period",
        "client_id",
        "region_id",
        "target_turnover_rate",
        "target_utilization_rate",
        "target_cost",
        "target_billable_hours",
        "source_file",
        "ingestion_timestamp",
        "bronze_load_id"
    )
)

# Preview the Gold goals fact table.
display(fact_goals_monthly_df.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


goals_fact_key,goal_record_id,date_key,client_key,region_key,period,client_id,region_id,target_turnover_rate,target_utilization_rate,target_cost,target_billable_hours,source_file,ingestion_timestamp,bronze_load_id
1,2026-01|C001|APAC,202601,1,1,2026-01,C001,APAC,0.056,0.8297,283777.28,3220.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
2,2026-01|C001|EMEA,202601,1,2,2026-01,C001,EMEA,0.0681,0.8336,293517.97,3327.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
3,2026-01|C001|LATAM,202601,1,3,2026-01,C001,LATAM,0.0205,0.8086,322642.43,6410.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
4,2026-01|C002|APAC,202601,2,1,2026-01,C002,APAC,0.0796,0.7263,249599.8,6019.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
5,2026-01|C002|EMEA,202601,2,2,2026-01,C002,EMEA,0.0572,0.8613,349859.86,4558.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
6,2026-01|C002|LATAM,202601,2,3,2026-01,C002,LATAM,0.0457,0.7866,224553.48,3479.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
7,2026-01|C002|NA,202601,2,4,2026-01,C002,NA,0.0692,0.7386,207959.34,2606.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
8,2026-01|C003|APAC,202601,3,1,2026-01,C003,APAC,0.0695,0.7372,318171.52,4917.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
9,2026-01|C003|EMEA,202601,3,2,2026-01,C003,EMEA,0.0606,0.7683,266511.27,3271.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3
10,2026-01|C003|LATAM,202601,3,3,2026-01,C003,LATAM,0.0536,0.746,306154.2,4967.0,goals.csv,2026-05-09T12:44:27.808Z,e6881281-d99b-4b4a-b7ed-846546bcd8f3


In [0]:
# Count records in Silver goals and Gold fact_goals_monthly.
silver_goals_count = silver_goals_df.count()
fact_goals_count = fact_goals_monthly_df.count()

print(f"Silver goals rows: {silver_goals_count}")
print(f"Gold fact_goals_monthly rows: {fact_goals_count}")

# Check that goals_fact_key is unique.
goals_fact_key_duplicates = (
    fact_goals_monthly_df
    .groupBy("goals_fact_key")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicated goals_fact_key values: {goals_fact_key_duplicates}")

# Check that goals_fact_key is not null.
goals_fact_key_null_count = (
    fact_goals_monthly_df
    .filter(F.col("goals_fact_key").isNull())
    .count()
)

print(f"Rows with null goals_fact_key: {goals_fact_key_null_count}")

# Check missing dimensional keys.
missing_date_key_count = fact_goals_monthly_df.filter(F.col("date_key").isNull()).count()
missing_client_key_count = fact_goals_monthly_df.filter(F.col("client_key").isNull()).count()
missing_region_key_count = fact_goals_monthly_df.filter(F.col("region_key").isNull()).count()

print(f"Rows without date_key: {missing_date_key_count}")
print(f"Rows without client_key: {missing_client_key_count}")
print(f"Rows without region_key: {missing_region_key_count}")

# Check that target rates are within expected ranges.
invalid_target_turnover_rate_count = (
    fact_goals_monthly_df
    .filter(
        (F.col("target_turnover_rate") < 0)
        | (F.col("target_turnover_rate") > 1)
    )
    .count()
)

invalid_target_utilization_rate_count = (
    fact_goals_monthly_df
    .filter(
        (F.col("target_utilization_rate") < 0)
        | (F.col("target_utilization_rate") > 1)
    )
    .count()
)

print(f"Rows with invalid target_turnover_rate: {invalid_target_turnover_rate_count}")
print(f"Rows with invalid target_utilization_rate: {invalid_target_utilization_rate_count}")

# Check that target values are not negative.
invalid_target_cost_count = (
    fact_goals_monthly_df
    .filter(F.col("target_cost") < 0)
    .count()
)

invalid_target_billable_hours_count = (
    fact_goals_monthly_df
    .filter(F.col("target_billable_hours") < 0)
    .count()
)

print(f"Rows with invalid target_cost: {invalid_target_cost_count}")
print(f"Rows with invalid target_billable_hours: {invalid_target_billable_hours_count}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Silver goals rows: 435
Gold fact_goals_monthly rows: 435
Duplicated goals_fact_key values: 0
Rows with null goals_fact_key: 0
Rows without date_key: 0
Rows without client_key: 0
Rows without region_key: 0
Rows with invalid target_turnover_rate: 0
Rows with invalid target_utilization_rate: 0
Rows with invalid target_cost: 0
Rows with invalid target_billable_hours: 0


# Save Gold Tables

## Objective

Persist all Gold dimensions and fact tables as Delta tables in Databricks.

## Tables created

- `dim_region`
- `dim_date`
- `dim_employee`
- `dim_client`
- `fact_workforce_monthly`
- `fact_goals_monthly`

## Notes

These Gold tables are designed for reporting and dashboard consumption.

In [0]:
# Save Gold DataFrames as managed Delta tables.
# overwriteSchema is enabled because Gold table structures may evolve during development.

dim_region_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("dim_region")
dim_date_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("dim_date")
dim_employee_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("dim_employee")
dim_client_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("dim_client")
fact_workforce_monthly_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("fact_workforce_monthly")
fact_goals_monthly_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("fact_goals_monthly")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


# Final Gold Row Counts

## Objective

Verify that all Gold tables were created and check their final row counts.

In [0]:
# Final row count validation for Gold tables.
# This is a quick sanity check to confirm that all tables were saved correctly.

gold_tables = [
    "dim_region",
    "dim_date",
    "dim_employee",
    "dim_client",
    "fact_workforce_monthly",
    "fact_goals_monthly",
]

for table in gold_tables:
    count = spark.table(table).count()
    print(f"{table}: {count} rows")

dim_region: 4 rows
dim_date: 12 rows
dim_employee: 493 rows
dim_client: 13 rows
fact_workforce_monthly: 3292 rows
fact_goals_monthly: 435 rows
